# PFAS Groundwater — HGT Encoder + Embedding Fusion + Stacking Ensemble (T1a)

**Task T1a**: predict EPA-2024 regulatory PFAS exceedance (binary) from context
features only — strict predictive mode, no PFAS measurement as input.

**This notebook is AUTONOMOUS** (CLAUDE.md §4): bootstraps `src/` and the versioned
dataset via `git clone` (no Google Drive, no gdown), installs PyTorch Geometric for
the Colab torch wheel, then runs the three-architecture experiment from
`src/hgt_fusion_stacking_t1.py` via a single call to `M.run(smoke=...)`.

**Three architectures under ONE spatial-block protocol:**
1. **HGT standalone** — multi-relational encoder over the well-well graph (two real
   edge types: `near` spatial k-NN cap 1.5 km, `same_subbasin_knn` intra-sub-basin
   k-NN cap 2 km) + classification head.
2. **Embedding fusion** — XGBoost on [tabular features (+) PCA-reduced HGT
   embeddings to 95% variance].
3. **Stacking** — meta-XGBoost over OOF predictions from {HGT, XGB-tabular,
   LightGBM-tabular} plus agreement / entropy meta-features.

**Anti-leak design:** one leave-one-block-out (LOBO) over 8 spatial KMeans blocks
produces per-well OOF arrays; a SECOND nested LOBO evaluates fusion and stacking;
cross-block edges asserted 0 per relation (C-SPAT.2/5); lat/lon never a feature
(C-LOC.1); thresholds from OOF only (C-THR). Evaluation at sampling (row) level,
comparable to the non-graph WALL (XGB spatial AUC 0.5878).

**Checkpoints** written to `experiments/hgt_fusion_stacking_t1/` after the expensive
spatial arm (metrics_incremental via `_write_metrics`), so a Colab disconnect does
not lose the spatial arm's work.

**Persistence cell** at the end: `files.download()` archive and/or `git push`
(no Drive — Colab workspace is EPHEMERAL).

---
**CRITICAL — push the code first:** `src/hgt_fusion_stacking_t1.py` and this notebook
are currently UNTRACKED in git. The `git clone` in Cell 3 only sees committed and pushed
files. You MUST run the following before opening this notebook on Colab:
```
git add src/hgt_fusion_stacking_t1.py notebooks/hgt_fusion_stacking_t1_colab.ipynb
git commit -m 'feat: HGT fusion stacking T1 module + Colab notebook'
git push
```
Then set `GIT_REF` below to the branch or commit SHA you just pushed.

---
> `SMOKE_TEST=True`: CPU sanity check (~1-3 min, 500-well subsample, 3 blocks, 15 epochs).  
> `SMOKE_TEST=False`: full GPU run — see estimated duration in Cell 0.

## Cell 0 — User parameters (read this before running)

In [ ]:
# ============================================================
# USER PARAMETERS — adjust before running
# ============================================================

SMOKE_TEST = False        # True = fast CPU sanity check (~1-3 min); False = full GPU run

# IMPORTANT: push src/hgt_fusion_stacking_t1.py + this notebook to the repo BEFORE
# running on Colab (see the markdown cell above). Set GIT_REF to the branch/commit
# that contains both files.
REPO_URL = "https://github.com/dnwiloic/pfas-gnn.git"
GIT_REF  = "main"        # branch name or full commit SHA — must include hgt_fusion_stacking_t1.py
DATA_PATH = "data/CA-PFAS-ASGWS.parquet"  # relative to repo root (versioned in repo)

EXP_DIR = "experiments/hgt_fusion_stacking_t1"

# Full-run HGT hyperparameters (match src/hgt_fusion_stacking_t1.py defaults).
# Change only for ablation.
FULL_N_BLOCKS    = 8     # spatial KMeans CV blocks
FULL_HIDDEN      = 64    # HGT hidden dim (also PCA input size)
FULL_LAYERS      = 2
FULL_DROPOUT     = 0.3
FULL_HEADS       = 4
FULL_MAX_EPOCHS  = 400
FULL_PATIENCE    = 50
FULL_LR          = 5e-3
FULL_WEIGHT_DECAY = 5e-4
FULL_PCA_VAR     = 0.95  # PCA variance threshold for embedding fusion
COMPUTE_DELTA    = True  # also run random-block CV to measure Δ(random − spatial)

SEED = 42

# ============================================================
# DURATION ESTIMATE (SMOKE_TEST=False, Colab T4 GPU)
# ============================================================
# Architecture breakdown:
#   Outer LOBO: 8 blocks x (HGT fold train + XGB + LGBM tabular) = 8 x 3 base fits
#   Nested LOBO for fusion + stacking: 8 meta fits each
#   PLUS delta arm (random blocks): same cost again
#
# Smoke test ran in ~63s on CPU for 500 wells / 3 blocks / 15 epochs.
# Scaling from smoke -> full (CPU):
#   scale_wells  = 11333 / 500  ~ 22.7x
#   scale_epochs = 400 / 15     ~ 26.7x
#   scale_blocks = 8 / 3        ~ 2.7x
#   (nonlinear: GNN scales sublinearly with nodes due to batch sampling)
# Conservative CPU estimate: 63s x 22.7 x (400/15) x (8/3) / 60 ~ 215 min per arm.
# With delta arm: ~430 min CPU -> ~2-4h on Colab T4 GPU (GPU ~10-15x speedup).
#
# Expected wall time on Colab T4 GPU: 2-4 hours.
# Checkpointing: metrics.json written after the spatial arm completes, before delta.
# ============================================================

print("Parameters set.")
print(f"  SMOKE_TEST : {SMOKE_TEST}")
print(f"  REPO_URL   : {REPO_URL}")
print(f"  GIT_REF    : {GIT_REF}")
print(f"  DATA_PATH  : {DATA_PATH}")
print(f"  EXP_DIR    : {EXP_DIR}")
if not SMOKE_TEST:
    print()
    print("  Full-run estimate (Colab T4 GPU): ~2-4 hours")
    print("  Checkpoint: metrics.json written after the spatial arm (mid-run safety net).")
    print("  Run MUST be preceded by: git add + commit + push of src/hgt_fusion_stacking_t1.py")

## Cell 1 — GPU detection & runtime versions

In [ ]:
import sys, platform, subprocess

print(f"Python  : {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass
print(f"IN_COLAB: {IN_COLAB}")

try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"torch   : {torch.__version__}  CUDA avail: {cuda_ok}")
    if cuda_ok:
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        print(f"CUDA    : {torch.version.cuda}")
    else:
        print("WARNING: no GPU detected.")
        if SMOKE_TEST:
            print("  SMOKE_TEST=True -> CPU run is expected and fine.")
        else:
            print("  SMOKE_TEST=False -> full run needs GPU.")
            print("  Go to: Runtime > Change runtime type > T4 GPU")
except ImportError:
    print("torch not yet installed (Colab base image usually has it).")

# Note: XGBoost and LightGBM tabular bases run on CPU (tree_method='hist').
# The GNN embedding step is the GPU-accelerated part.
# For SMOKE_TEST=True a CPU runtime (High-RAM) avoids GPU quota warnings.

## Cell 2 — Install pinned dependencies + verify imports

PyG compiled extensions must match `torch.__version__` + CUDA tag.
We detect both at runtime and install from the matching wheel index.
This cell is idempotent: if PyG is already importable it skips the install.

In [ ]:
import subprocess, sys

# --- PyTorch Geometric (must match runtime torch + CUDA) ---
def ensure_pyg():
    try:
        import torch_geometric
        print(f"torch_geometric already present: {torch_geometric.__version__}")
        return
    except ImportError:
        pass
    import torch
    tv = torch.__version__.split("+")[0]
    cuda = torch.version.cuda
    tag = f"cu{cuda.replace('.', '')}" if (cuda and torch.cuda.is_available()) else "cpu"
    idx = f"https://data.pyg.org/whl/torch-{tv}+{tag}.html"
    print(f"Installing torch_geometric for torch={tv}, device={tag}")
    print(f"  Wheel index: {idx}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "-f", idx], check=True)
    # Optional compiled helpers (scatter/sparse) — ignore failure on CPU
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_scatter", "torch_sparse", "-f", idx])

ensure_pyg()

# Verify PyG import and confirm HGTConv is available (needed by hgt_fusion_stacking_t1)
import torch_geometric
from torch_geometric.nn import HGTConv  # noqa — the multi-relational encoder core
print(f"PyG: {torch_geometric.__version__}  HGTConv import OK")

# --- XGBoost (Colab usually has it; install pinned version if absent) ---
try:
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__}")
except ImportError:
    print("xgboost not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0"],
                   check=True)
    import xgboost as xgb
    print(f"xgboost : {xgb.__version__} (just installed)")

# --- LightGBM (hgt_fusion_stacking_t1 falls back to sklearn if absent; install anyway) ---
try:
    import lightgbm as lgb
    print(f"lightgbm: {lgb.__version__}")
except ImportError:
    print("lightgbm not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm>=4.0"],
                   check=True)
    import lightgbm as lgb
    print(f"lightgbm: {lgb.__version__} (just installed)")

# --- Other deps ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyarrow>=14.0", "pyyaml>=6.0", "scikit-learn>=1.4"], check=True)

print("All dependencies satisfied.")

## Cell 3 — Clone repo (brings src/ AND data/ — no Drive, no gdown)

The dataset `data/CA-PFAS-ASGWS.parquet` is **versioned in the repo** and arrives
automatically with the clone. This cell is idempotent: if the repo dir already
exists it only checkouts the requested ref.

In [ ]:
import os, sys, importlib.util, pathlib

REPO_DIR = "/content/pfas-gnn" if IN_COLAB else os.getcwd()

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Checking out {GIT_REF} ...")
    subprocess.run(["git", "-C", REPO_DIR, "checkout", GIT_REF], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"workdir : {os.getcwd()}")

# ---- Anti-stale-code guard ----
# Verify that the cloned repo at GIT_REF actually contains hgt_fusion_stacking_t1.py.
# If this fails, you must: git add src/hgt_fusion_stacking_t1.py && git commit && git push
# BEFORE running on Colab, then update GIT_REF to point to that commit.
MODULE_PATH = pathlib.Path(REPO_DIR) / "src" / "hgt_fusion_stacking_t1.py"
assert MODULE_PATH.exists(), (
    f"FATAL: src/hgt_fusion_stacking_t1.py not found in the cloned repo at GIT_REF={GIT_REF!r}.\n"
    "You must commit and push the module BEFORE running this notebook on Colab:\n"
    "  git add src/hgt_fusion_stacking_t1.py notebooks/hgt_fusion_stacking_t1_colab.ipynb\n"
    "  git commit -m 'feat: HGT fusion stacking T1 module + Colab notebook'\n"
    "  git push\n"
    "Then set GIT_REF to the resulting branch or commit SHA and re-run this cell."
)

# Verify key symbols are present in the source (catches truncated/wrong commit)
_src_text = MODULE_PATH.read_text()
for symbol in ["def run(", "def build_oof_backbone(", "def fusion_oof_proba(",
               "def stacking_oof_proba(", "class OOFArrays"]:
    assert symbol in _src_text, (
        f"Symbol '{symbol}' not found in src/hgt_fusion_stacking_t1.py — "
        f"the repo at GIT_REF={GIT_REF!r} may be pointing to an incomplete commit. "
        "Push the complete module and update GIT_REF."
    )
    print(f"  {symbol!r:50s} OK")

# Verify dataset present
assert os.path.exists(DATA_PATH), (
    f"FATAL: dataset missing at {DATA_PATH}.\n"
    "The parquet file should be versioned in the repo at data/CA-PFAS-ASGWS.parquet.\n"
    "Check REPO_URL and GIT_REF — the clone may have failed or the file is not committed."
)
print(f"\ndataset : {DATA_PATH}  ({os.path.getsize(DATA_PATH)//1024} KB)")
print("Code + dataset guard PASSED.")

## Cell 4 — Load dataset & integrity check

Expected shape: **46338 rows x 201 columns**, **11333 unique wells**.  
Required columns: `gm_well_id`, `latitude`, `longitude`, `PFOA_ngL`, `sgma_subbasin_name`.  
Any mismatch triggers an explicit stop — no silent failure downstream.

In [ ]:
import pandas as pd

# Expected dataset dimensions
EXPECTED_ROWS  = 46338
EXPECTED_COLS  = 201
EXPECTED_WELLS = 11333
KEY_COLS = ["gm_well_id", "latitude", "longitude", "PFOA_ngL", "sgma_subbasin_name"]

df_probe = pd.read_parquet(DATA_PATH)
n_rows, n_cols = df_probe.shape
n_wells = df_probe["gm_well_id"].nunique()
missing_keys = [c for c in KEY_COLS if c not in df_probe.columns]

print(f"Shape  : {n_rows} rows x {n_cols} cols")
print(f"Wells  : {n_wells} unique")
print(f"Memory : {df_probe.memory_usage(deep=True).sum() // 1024 // 1024} MB")

if missing_keys:
    raise RuntimeError(
        f"FATAL: key columns missing from dataset: {missing_keys}.\n"
        f"Check DATA_PATH={DATA_PATH!r} and GIT_REF={GIT_REF!r}.\n"
        "The parquet must contain: " + ", ".join(KEY_COLS)
    )
print(f"Key columns: {KEY_COLS} — all present")

if not SMOKE_TEST:
    # Strict shape check for the full run to catch wrong dataset versions
    if n_rows != EXPECTED_ROWS:
        raise RuntimeError(
            f"FATAL: row count mismatch — got {n_rows}, expected {EXPECTED_ROWS}.\n"
            f"Clone may be stale or DATA_PATH={DATA_PATH!r} points to the wrong file.\n"
            f"Check GIT_REF={GIT_REF!r}."
        )
    if n_wells != EXPECTED_WELLS:
        raise RuntimeError(
            f"FATAL: well count mismatch — got {n_wells}, expected {EXPECTED_WELLS}.\n"
            f"Check GIT_REF={GIT_REF!r} and DATA_PATH={DATA_PATH!r}."
        )
    print(f"Integrity check PASSED: {n_rows} x {n_cols}, {n_wells} wells.")
else:
    print(f"SMOKE_TEST=True — shape check relaxed (got {n_rows} x {n_cols}, {n_wells} wells).")
    print("  The module will subsample to 500 wells internally.")

del df_probe  # free memory; the module reloads via src.data.load()

## Cell 5 — Smoke-test (`SMOKE_TEST=True` only)

Calls `M.run(smoke=True)` which:
- subsamples 500 wells, uses 3 spatial blocks, 15 HGT epochs (patience 6),
- runs all three architectures end-to-end (OOF backbone + fusion + stacking),
- asserts cross-block edges = 0, checks loss is finite, writes artefacts to
  `experiments/hgt_fusion_stacking_t1/`.

When `SMOKE_TEST=False` this cell is skipped.

In [ ]:
import time, json, math
from pathlib import Path

if SMOKE_TEST:
    print("=" * 65)
    print("SMOKE-TEST: 500 wells / 3 blocks / 15 epochs — CPU run")
    print("=" * 65)

    from src import hgt_fusion_stacking_t1 as M

    t0 = time.time()
    smoke_out = M.run(
        smoke=True,
        write=True,
        exp_dir=EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_smoke = time.time() - t0

    # ---- end-to-end assertions ----
    assert "spatial" in smoke_out, "SMOKE FAIL: 'spatial' key missing from output"
    sp = smoke_out["spatial"]
    assert sp["n_cross_block_total"] == 0, (
        f"LEAK DETECTED: {sp['n_cross_block_total']} cross-block edges remain! "
        "This is a spatial-leakage violation (C-SPAT.2/5)."
    )
    archs = sp["architectures"]
    for arch_name in ["hgt_standalone", "embedding_fusion", "stacking"]:
        assert arch_name in archs, f"SMOKE FAIL: architecture '{arch_name}' missing"
        auc = archs[arch_name]["metrics"]["roc_auc"]
        assert math.isfinite(auc), f"SMOKE FAIL: AUC is not finite for {arch_name}"
        brier = archs[arch_name]["metrics"]["brier"]
        assert math.isfinite(brier) and brier < 1.0, (
            f"SMOKE FAIL: Brier score invalid for {arch_name}: {brier}"
        )
    # Check artefacts written
    assert Path(EXP_DIR, "metrics.json").exists(), (
        f"SMOKE FAIL: metrics.json not written to {EXP_DIR}/"
    )

    print()
    print(f"Smoke test elapsed: {elapsed_smoke:.0f}s ({elapsed_smoke/60:.1f} min)")
    print(f"Cross-block edges : {sp['n_cross_block_total']}  (must be 0) OK")
    print()
    print("Smoke architecture results (not representative — tiny subsample):")
    for arch_name in ["hgt_standalone", "embedding_fusion", "stacking"]:
        m = archs[arch_name]["metrics"]
        print(f"  {arch_name:<30s}  AUC={m['roc_auc']:.4f}  "
              f"F1={m['f1']:.4f}  Brier={m['brier']:.4f}")
    print()

    # ---- full-run duration estimate ----
    # Smoke: 500 wells, 3 blocks, 15 epochs
    # Full:  ~11333 wells, 8 blocks, 400 epochs, PLUS delta (random) arm
    # HGT scale (non-linear due to neighbour sampling): conservative x well_ratio
    smoke_n_wells = smoke_out["meta"].get("n_wells", 500)
    scale_wells   = 11333 / max(smoke_n_wells, 1)
    scale_epochs  = 400   / 15
    scale_blocks  = 8     / 3
    scale_delta   = 2     # spatial + random arms
    # Conservative wall time (sublinear in wells, but blocks and epochs are multiplicative)
    est_s = elapsed_smoke * scale_epochs * scale_blocks * scale_delta
    est_gpu_s = est_s / 10   # T4 GPU ~10x CPU speedup (conservative)
    print(f"Full-run wall-time estimate (CPU)  : {est_s/60:.0f} min ({est_s/3600:.1f} h)")
    print(f"Full-run wall-time estimate (T4 GPU): {est_gpu_s/60:.0f} min ({est_gpu_s/3600:.1f} h)")
    if est_gpu_s > 4 * 3600:
        print()
        print("WARNING: estimated GPU time > 4 hours. Colab sessions typically time out")
        print("  after ~12 hours. The checkpoint after the spatial arm provides a safety")
        print("  net, but consider setting COMPUTE_DELTA=False to halve the budget.")
    else:
        print("  -> Fits comfortably in a Colab T4 session. Proceed to the full run.")

    print()
    print("SMOKE-TEST PASSED. Set SMOKE_TEST=False and re-run for the full GPU run.")

else:
    print("SMOKE_TEST=False — smoke cell skipped. Proceed to Cell 6 for the full run.")

## Cell 6 — Full run (`SMOKE_TEST=False`)

Calls `M.run(smoke=False)` which runs:
1. One leave-one-block-out (8 spatial KMeans blocks) producing per-well OOF arrays
   for HGT, XGB-tabular, and LightGBM-tabular (the backbone).
2. Nested LOBO for embedding fusion (PCA-to-95%-variance + XGB) and stacking
   (meta-XGB over three OOF bases + agreement/entropy features).
3. Same regime with RANDOM blocks (the delta arm) if `COMPUTE_DELTA=True`.
4. Paired Nadeau-Bengio + Wilcoxon tests vs the in-run XGB-tabular wall.

**Checkpoint:** `experiments/hgt_fusion_stacking_t1/metrics.json` is written after
the spatial arm completes (before the delta arm). If the runtime disconnects during
the delta arm you keep all spatial results.

All artefacts (`metrics.json`, `REPORT.md`, `config.yaml`) are written under
`experiments/hgt_fusion_stacking_t1/` inside the cloned workspace.

In [ ]:
import time, json
import numpy as np
from pathlib import Path

if not SMOKE_TEST:
    print("=" * 65)
    print("FULL RUN: HGT fusion stacking T1a — spatial CV (8 blocks)")
    print("=" * 65)
    print(f"  hidden={FULL_HIDDEN}  layers={FULL_LAYERS}  heads={FULL_HEADS}  "
          f"dropout={FULL_DROPOUT}")
    print(f"  max_epochs={FULL_MAX_EPOCHS}  patience={FULL_PATIENCE}  "
          f"lr={FULL_LR}  wd={FULL_WEIGHT_DECAY}")
    print(f"  n_blocks={FULL_N_BLOCKS}  pca_var={FULL_PCA_VAR}  "
          f"compute_delta={COMPUTE_DELTA}  seed={SEED}")
    print()
    print("Checkpoint: metrics.json written after spatial arm (mid-run safety net).")
    print("Progress printed per block by verbose=True inside the module.")
    print()

    from src import hgt_fusion_stacking_t1 as M

    t0 = time.time()
    full_out = M.run(
        smoke=False,
        n_blocks=FULL_N_BLOCKS,
        hidden=FULL_HIDDEN,
        layers=FULL_LAYERS,
        dropout=FULL_DROPOUT,
        heads=FULL_HEADS,
        max_epochs=FULL_MAX_EPOCHS,
        patience=FULL_PATIENCE,
        lr=FULL_LR,
        weight_decay=FULL_WEIGHT_DECAY,
        pca_var=FULL_PCA_VAR,
        compute_delta=COMPUTE_DELTA,
        write=True,
        exp_dir=EXP_DIR,
        seed=SEED,
        verbose=True,
    )
    elapsed_full = time.time() - t0

    # Verify anti-leak assertion held for the full run
    assert full_out["spatial"]["n_cross_block_total"] == 0, (
        "LEAK DETECTED in full run: cross-block edges > 0. "
        "Do NOT use these results — spatial leakage violates C-SPAT.2/5."
    )

    print(f"\nFull run completed in {elapsed_full:.0f}s ({elapsed_full/60:.1f} min)")
    print(f"Cross-block edges: {full_out['spatial']['n_cross_block_total']}  (must be 0) OK")
    print()

else:
    full_out = None
    print("SMOKE_TEST=True — full run cell skipped.")

## Cell 7 — Display results

Reads `experiments/hgt_fusion_stacking_t1/metrics.json` and renders the
comparison table plus the REPORT.md text. Reading from disk ensures the
display matches what was actually persisted (even if the full run was done
in a previous session and `full_out` is not in scope).

In [ ]:
import json
from pathlib import Path
import numpy as np

metrics_path = Path(EXP_DIR) / "metrics.json"
report_path  = Path(EXP_DIR) / "REPORT.md"

if not metrics_path.exists():
    print(f"metrics.json not found at {metrics_path}.")
    print("Run Cell 5 (smoke) or Cell 6 (full run) first.")
else:
    out = json.loads(metrics_path.read_text())
    meta = out["meta"]
    sp   = out["spatial"]
    A    = sp["architectures"]
    cmp  = out.get("comparison", {})

    print(f"smoke={meta['smoke']}  seed={meta['seed']}  blocks={meta['n_blocks']}  "
          f"features={meta['n_features']}  elapsed={meta.get('elapsed_s', float('nan')):.0f}s")
    print(f"Cross-block edges (spatial arm): {sp['n_cross_block_total']}")
    print()

    WALL_XGB = out["meta"].get("wall_xgb_spatial_auc", 0.5878)
    WALL_RF  = out["meta"].get("wall_rf_spatial_auc",  0.6009)

    # ---- Architecture comparison table ----
    header = f"{'Architecture':<38} {'AUC':>7}  {'[95%CI]':^16}  {'F1':>6}  "\
             f"{'PR-AUC':>7}  {'Bal.Acc':>7}  {'Brier':>7}  {'ECE':>6}"
    print(header)
    print("-" * len(header))

    def _row(label, arch_dict):
        m  = arch_dict["metrics"]
        ci = arch_dict["auc_ci95"]
        return (f"{label:<38} {m['roc_auc']:>7.4f}  "
                f"[{ci['ci_low']:.3f},{ci['ci_high']:.3f}]  "
                f"{m['f1']:>6.4f}  {m['pr_auc']:>7.4f}  "
                f"{m['balanced_accuracy']:>7.4f}  {m['brier']:>7.4f}  "
                f"{m.get('ece', float('nan')):>6.4f}")

    print(_row("HGT standalone", A["hgt_standalone"]))
    print(_row("Embedding fusion (XGB+PCA-HGT)", A["embedding_fusion"]))
    print(_row("Stacking (HGT+XGB+LGBM meta)", A["stacking"]))
    refs = sp.get("base_references", {})
    if "xgb_tabular" in refs:
        print(_row("XGB-tabular (in-run wall)", refs["xgb_tabular"]))
    if "lgbm_tabular" in refs:
        print(_row("LGBM-tabular (ref)", refs["lgbm_tabular"]))

    fus = A["embedding_fusion"]
    print()
    print(f"PCA-to-{int(fus['pca_variance_target']*100)}%-var: "
          f"{fus['pca_n_components_mean']:.1f} components on average "
          f"(per fold: {fus['pca_n_components_per_fold']}) "
          f"out of {meta['hidden']} HGT-embed dims.")

    # ---- Delta arm (if computed) ----
    if "random" in out and "delta_random_minus_spatial" in out:
        d = out["delta_random_minus_spatial"]
        rA = out["random"]["architectures"]
        print()
        print("Delta(random - spatial) — spatial-leakage inflation:")
        for key, lab in [("hgt_standalone", "HGT standalone"),
                         ("embedding_fusion", "Embedding fusion"),
                         ("stacking", "Stacking")]:
            sp_auc = A[key]["metrics"]["roc_auc"]
            rd_auc = rA[key]["metrics"]["roc_auc"]
            print(f"  {lab:<30}  spatial={sp_auc:.4f}  random={rd_auc:.4f}  "
                  f"delta={d.get(key, float('nan')):+.4f}")

    # ---- Verdict vs WALL ----
    if cmp:
        print()
        print(f"Committed WALL: XGB={WALL_XGB:.4f}  RF={WALL_RF:.4f}")
        print(f"In-run XGB wall: {cmp.get('in_run_xgb_wall_auc_mean', float('nan')):.4f}")
        print()
        print("Paired-test verdicts (Nadeau-Bengio + Wilcoxon, 8 spatial folds):")
        for arch_name, rec in cmp.get("by_architecture", {}).items():
            nb_p = rec.get("nadeau_bengio", {}).get("p", float("nan"))
            wc_p = rec.get("wilcoxon",      {}).get("p", float("nan"))
            print(f"  {arch_name:<30}  "
                  f"gain_vs_inrun={rec.get('gain_vs_in_run_wall', float('nan')):+.4f}  "
                  f"gain_vs_committed={rec.get('gain_vs_committed_wall', float('nan')):+.4f}  "
                  f"NB_p={nb_p:.4f}  WC_p={wc_p:.4f}  "
                  f"verdict={rec.get('verdict', 'n/a')}")

    # ---- REPORT.md ----
    if report_path.exists():
        print()
        print("=" * 65)
        print("REPORT.md")
        print("=" * 65)
        print(report_path.read_text())

## Cell 8 — Persist outputs

> **WARNING: the Colab workspace is EPHEMERAL.** All files in `/content/pfas-gnn/`
> are **permanently lost when the runtime disconnects or times out**. Without running
> this cell your results (`metrics.json`, `REPORT.md`, `config.yaml`) will be gone.

**Option A (quick):** `files.download()` creates a local zip archive on your machine.

**Option B (recommended):** `git commit + push` writes results back to the repo so
they survive and are versioned. Requires push access to `REPO_URL`.

Run at least one option before closing the session.

In [ ]:
import shutil, os
from pathlib import Path

exp_path = Path(EXP_DIR)

if not exp_path.exists():
    print(f"No outputs found at {EXP_DIR}. Run Cell 5 or Cell 6 first.")
else:
    files_found = sorted(exp_path.rglob("*"))
    print(f"Outputs in {EXP_DIR}/ ({len(files_found)} files/dirs):")
    for p in files_found:
        if p.is_file():
            print(f"  {p.relative_to(exp_path)}  ({p.stat().st_size // 1024} KB)")

    # ==== Option A: download zip archive ====
    archive_name = "hgt_fusion_stacking_t1_outputs"
    archive_path = shutil.make_archive(archive_name, "zip",
                                       root_dir=str(exp_path.parent),
                                       base_dir=exp_path.name)
    archive_size = os.path.getsize(archive_path) // 1024
    print(f"\nOption A — Archive created: {archive_path} ({archive_size} KB)")
    if IN_COLAB:
        from google.colab import files
        files.download(archive_path)
        print("  Download triggered — save this zip to your local machine.")
    else:
        print(f"  (Not in Colab — archive at {os.path.abspath(archive_path)})")

    # ==== Option B: git commit + push ====
    # Uncomment the block below and fill GIT_USER / GIT_EMAIL if needed.
    # This pushes the experiment outputs back to REPO_URL so they are versioned.
    #
    # import subprocess
    # # (Optional) set identity if not configured in the cloned repo:
    # subprocess.run(["git", "config", "user.email", "your@email.com"], cwd=REPO_DIR)
    # subprocess.run(["git", "config", "user.name", "Your Name"],       cwd=REPO_DIR)
    # subprocess.run(["git", "add",
    #     "experiments/hgt_fusion_stacking_t1/metrics.json",
    #     "experiments/hgt_fusion_stacking_t1/REPORT.md",
    #     "experiments/hgt_fusion_stacking_t1/config.yaml",
    # ], cwd=REPO_DIR, check=True)
    # import json as _json
    # _smoke_flag = "smoke" if SMOKE_TEST else "full"
    # subprocess.run(["git", "commit", "-m",
    #     f"results(hgt_fusion_stacking_t1): {_smoke_flag} run — 3-arch T1a from Colab GPU"
    # ], cwd=REPO_DIR, check=True)
    # subprocess.run(["git", "push"], cwd=REPO_DIR, check=True)
    # print("Option B — Results committed and pushed to", REPO_URL)

    print()
    print("REMINDER: without Option A download OR Option B push, ALL outputs")
    print("are permanently lost when this Colab runtime disconnects or times out.")